
# Script 3 Full EEG Processing, Quality Control and Feature Extraction

**Dataset:** 22 participants × 3 environmental scenes = **66 EEG recordings**  
Expected file pattern: `Participant_Scene_EEG.csv` (e.g. `IC01_BG_EEG.csv`)

This notebook is the final full-dataset EEG pipeline. It retains the pilot preprocessing and robust peak-to-peak/maximum-step QC framework, with an additional flat-signal safeguard, and scales the workflow to all 66 recordings.

## Workflow

1. Discover and validate the complete 66-file dataset.
2. Remove the first 15 s of each recording.
3. Apply the fixed 1–30 Hz Butterworth band-pass filter.
4. Split filtered CH1/CH2 signals into non-overlapping 4 s epochs.
5. Apply the robust automatic QC rules:
   - peak-to-peak amplitude;
   - maximum sample-to-sample step;
   - explicit rejection of flat epochs;
   - thresholds = median + 6 × robust SD, with a floor of 3 × median.
6. Create reason-coloured QC overviews.
7. Build an independent review queue and detailed plots for **high/medium review-priority** epochs.
8. Apply any decisions stored in the external decision CSV and retain only epochs clean in **both ears**.
9. Estimate spectral power using Welch PSD:
   - theta: 4 ≤ f < 8 Hz
   - alpha: 8 ≤ f < 13 Hz
   - beta: 13 ≤ f ≤ 30 Hz
10. Export recording-level, epoch-level, PSD and QC tables.
11. Once mandatory review is complete, create the final Script 3 EEG/QC figures.
12. Export a compact **Script 4 input table** with one row per participant × scene.

### Important separation from Script 4

Script 3 does **not** perform the final environmental inferential model. It produces clean EEG features and EEG-specific descriptive/QC figures. Script 4 should merge the scene-level EEG output with scene-level ECG and order/environment records, while keeping participant-level pre/post POMS change at its correct participant-level granularity.


In [1]:

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import trapezoid
from IPython.display import display

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## 1. User settings — normally only the data path needs changing

In [2]:

# =========================================================
# 1. USER SETTINGS
# =========================================================

# New full dataset folder.
DATA_DIR = Path(r"C:\Users\Harvey McWalter\Desktop\EEG_Data_22")

OUTPUT_DIR = DATA_DIR / "Script3_EEG_Results"
QC_OVERVIEW_DIR = OUTPUT_DIR / "qc_overviews"
REVIEW_PLOT_DIR = OUTPUT_DIR / "review_plots"
FIGURE_DIR = OUTPUT_DIR / "final_figures"

for folder in [OUTPUT_DIR, QC_OVERVIEW_DIR, REVIEW_PLOT_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Dataset structure expected from the final experiment.
EXPECTED_PARTICIPANTS = (
    [f"IC{i:02d}" for i in range(1, 16)]
    + [f"N{i:02d}" for i in range(1, 8)]
)
EXPECTED_SCENES = ["BG", "G", "GR"]
EXPECTED_NUMBER_OF_FILES = len(EXPECTED_PARTICIPANTS) * len(EXPECTED_SCENES)

FS = 500
RIGHT_EEG_COL = "CH1"
LEFT_EEG_COL = "CH2"

SCENE_NAMES = {
    "BG": "Blue-Green",
    "G": "Green",
    "GR": "Grey",
}
SCENE_SORT_ORDER = {"BG": 0, "G": 1, "GR": 2}
SCENE_COLORS = {
    "BG": "#2a9d8f",
    "G": "#55a868",
    "GR": "#7f7f7f",
}

# Frozen preprocessing settings from the pilot.
EEG_BAND = (1.0, 30.0)
TOTAL_POWER_BAND = (4.0, 30.0)

# Exact feature definitions:
# theta 4 <= f < 8, alpha 8 <= f < 13, beta 13 <= f <= 30.
BAND_DEFINITIONS = {
    "theta": {"low": 4.0, "high": 8.0, "include_high": True},
    "alpha": {"low": 8.0, "high": 13.0, "include_high": True},
    "beta":  {"low": 13.0, "high": 30.0, "include_high": True},
}

EPOCH_SECONDS = 4.0
FILTER_ORDER = 4
START_TRIM_SECONDS = 15.0
END_TRIM_SECONDS = 0.0
V_TO_UV = 1_000_000
NUMBER_OF_MADS = 6.0
WELCH_WINDOW_SECONDS = 2.0

# Review rules frozen from the pilot workflow:
# step-only = high review priority;
# PtP-only = medium review priority;
# both rules = low review priority because the automatic rejection is already strong.
REVIEW_PRIORITY_MAP = {
    "step_only": "high",
    "ptp_only": "medium",
    "both": "low",
    "flat": "low",
}
MANDATORY_REVIEW_PRIORITIES = {"high", "medium"}

DECISION_FILE = OUTPUT_DIR / "script3_manual_review_decisions.csv"

# Blank manual_decision = accept the automatic result.
# To show that a high/medium candidate was independently reviewed but the auto result was accepted,
# enter reviewer initials/name and leave manual_decision blank.
SAVE_QC_OVERVIEWS = True
GENERATE_REVIEW_PLOTS_FOR_PENDING = True
REQUIRE_REVIEW_BEFORE_FINAL_FIGURES = True

PLOT_DPI = 300
SCRIPT_VERSION = "Script3_v1_2026-08-09"

print("Data folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)
print("Expected recordings:", EXPECTED_NUMBER_OF_FILES)



Data folder: C:\Users\Harvey McWalter\Desktop\EEG_Data_22
Output folder: C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results
Expected recordings: 66


## 2. File discovery, validation and manual-decision loader

In [3]:

# =========================================================
# 2. FILE / DECISION HELPERS
# =========================================================

def parse_filename(file_path: Path) -> dict:
    pattern = re.compile(
        r"^(?P<participant>[A-Za-z]+\d+)_(?P<scene>BG|GR|G)_EEG$",
        flags=re.IGNORECASE,
    )
    match = pattern.fullmatch(file_path.stem)
    if match is None:
        raise ValueError(
            f"Filename does not match Participant_Scene_EEG.csv: {file_path.name}"
        )

    participant = match.group("participant").upper()
    scene_code = match.group("scene").upper()

    return {
        "participant": participant,
        "scene_code": scene_code,
        "scene_name": SCENE_NAMES[scene_code],
    }


def discover_and_validate_input_files(data_dir: Path):
    if not data_dir.exists():
        raise FileNotFoundError(f"Data folder not found: {data_dir}")

    candidate_files = []
    for path in data_dir.glob("*_EEG.csv"):
        try:
            info = parse_filename(path)
            candidate_files.append((path, info))
        except ValueError:
            # Ignore unrelated CSV files.
            pass

    if not candidate_files:
        raise FileNotFoundError(f"No valid *_EEG.csv files found in: {data_dir}")

    actual_keys = [(info["participant"], info["scene_code"]) for _, info in candidate_files]

    if len(actual_keys) != len(set(actual_keys)):
        duplicate_keys = pd.Series(actual_keys)[pd.Series(actual_keys).duplicated(keep=False)]
        raise ValueError(f"Duplicate participant-scene files found: {duplicate_keys.tolist()}")

    expected_keys = {
        (participant, scene)
        for participant in EXPECTED_PARTICIPANTS
        for scene in EXPECTED_SCENES
    }
    actual_key_set = set(actual_keys)

    missing = sorted(expected_keys - actual_key_set)
    unexpected = sorted(actual_key_set - expected_keys)

    if missing or unexpected or len(candidate_files) != EXPECTED_NUMBER_OF_FILES:
        message = [
            "Full-dataset validation failed.",
            f"Expected {EXPECTED_NUMBER_OF_FILES} files; found {len(candidate_files)}.",
        ]
        if missing:
            message.append(f"Missing participant-scene files: {missing}")
        if unexpected:
            message.append(f"Unexpected participant-scene files: {unexpected}")
        raise ValueError("\n".join(message))

    def sort_key(item):
        path, info = item
        participant_order = EXPECTED_PARTICIPANTS.index(info["participant"])
        return participant_order, SCENE_SORT_ORDER[info["scene_code"]]

    return [item[0] for item in sorted(candidate_files, key=sort_key)]


def load_manual_decisions(decision_file: Path):
    required = [
        "participant",
        "scene_code",
        "channel",
        "epoch",
        "manual_decision",
    ]
    optional = ["review_note", "reviewer"]

    if not decision_file.exists():
        return pd.DataFrame(columns=required + optional)

    table = pd.read_csv(decision_file, keep_default_na=False)

    missing = [column for column in required if column not in table.columns]
    if missing:
        raise KeyError(f"Decision file is missing columns: {missing}")

    table["participant"] = table["participant"].astype(str).str.strip().str.upper()
    table["scene_code"] = table["scene_code"].astype(str).str.strip().str.upper()
    table["channel"] = table["channel"].astype(str).str.strip().str.lower()
    table["epoch"] = pd.to_numeric(table["epoch"], errors="raise").astype(int)
    table["manual_decision"] = (
        table["manual_decision"].astype(str).str.strip().str.lower()
    )

    invalid_decisions = ~table["manual_decision"].isin(["", "keep", "reject"])
    if invalid_decisions.any():
        display(table.loc[invalid_decisions])
        raise ValueError("manual_decision must be blank, keep, or reject.")

    for column in optional:
        if column not in table.columns:
            table[column] = ""
        table[column] = table[column].astype(str).str.strip()

    duplicate = table.duplicated(
        subset=["participant", "scene_code", "channel", "epoch"],
        keep=False,
    )
    if duplicate.any():
        display(table.loc[duplicate])
        raise ValueError("Duplicate manual-decision keys were found.")

    return table


INPUT_FILES = discover_and_validate_input_files(DATA_DIR)
MANUAL_DECISIONS = load_manual_decisions(DECISION_FILE)

print(f"Validated {len(INPUT_FILES)} recordings.")
print("Participants:", len(EXPECTED_PARTICIPANTS))
print("Scenes per participant:", len(EXPECTED_SCENES))
print("Existing manual-decision rows:", len(MANUAL_DECISIONS))



Validated 66 recordings.
Participants: 22
Scenes per participant: 3
Existing manual-decision rows: 498


## 3. Common preprocessing, QC and spectral functions

In [4]:

# =========================================================
# 3. COMMON PROCESSING FUNCTIONS
# =========================================================

def load_recording(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_csv(file_path, low_memory=False)
    df.columns = df.columns.astype(str).str.strip()

    required_columns = [RIGHT_EEG_COL, LEFT_EEG_COL]
    missing_columns = [c for c in required_columns if c not in df.columns]
    if missing_columns:
        raise KeyError(
            f"{file_path.name}: missing EEG columns {missing_columns}. "
            f"Available columns: {df.columns.tolist()}"
        )

    for column in required_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    missing_values = df[required_columns].isna().sum()
    if missing_values.sum() > 0:
        raise ValueError(
            f"{file_path.name}: NaN/non-numeric EEG samples found:\n{missing_values}"
        )

    return df


def bandpass_filter(x, fs, lowcut, highcut, order=4):
    x = np.asarray(x, dtype=float)
    if x.ndim != 1:
        raise ValueError("Input signal must be one-dimensional.")
    if not np.all(np.isfinite(x)):
        raise ValueError("Signal contains NaN or infinite values.")

    sos = signal.butter(
        N=order,
        Wn=[lowcut, highcut],
        btype="bandpass",
        fs=fs,
        output="sos",
    )
    return signal.sosfiltfilt(sos, x)


def create_epochs(x, fs, epoch_seconds):
    x = np.asarray(x, dtype=float)
    samples_per_epoch = int(round(fs * epoch_seconds))
    number_of_epochs = len(x) // samples_per_epoch

    if number_of_epochs < 1:
        raise ValueError("Recording is shorter than one complete epoch.")

    usable_length = number_of_epochs * samples_per_epoch
    return x[:usable_length].reshape(number_of_epochs, samples_per_epoch)


def robust_upper_threshold(values, number_of_mads=6.0):
    values = np.asarray(values, dtype=float)
    median_value = np.median(values)
    mad = np.median(np.abs(values - median_value))
    robust_sd = 1.4826 * mad

    if robust_sd == 0:
        return median_value * 3 if median_value > 0 else np.inf

    threshold = median_value + number_of_mads * robust_sd
    return max(threshold, median_value * 3)


def neighbour_median(values):
    values = np.asarray(values, dtype=float)
    previous = np.concatenate([[np.nan], values[:-1]])
    following = np.concatenate([values[1:], [np.nan]])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanmedian(np.vstack([previous, following]), axis=0)


def assess_epoch_quality(epochs, epoch_seconds, number_of_mads=6.0):
    epochs = np.asarray(epochs, dtype=float)
    if epochs.ndim != 2:
        raise ValueError("Epoch data must be a 2D array.")

    peak_to_peak = np.ptp(epochs, axis=1)
    maximum_step = np.max(np.abs(np.diff(epochs, axis=1)), axis=1)

    ptp_threshold = robust_upper_threshold(peak_to_peak, number_of_mads)
    step_threshold = robust_upper_threshold(maximum_step, number_of_mads)

    finite_mask = np.all(np.isfinite(epochs), axis=1)
    non_flat_mask = peak_to_peak > 0
    ptp_clean_mask = peak_to_peak <= ptp_threshold
    step_clean_mask = maximum_step <= step_threshold

    clean_mask = finite_mask & non_flat_mask & ptp_clean_mask & step_clean_mask

    quality_table = pd.DataFrame({
        "epoch": np.arange(len(epochs)),
        "start_time_s": np.arange(len(epochs)) * epoch_seconds,
        "end_time_s": (np.arange(len(epochs)) + 1) * epoch_seconds,
        "peak_to_peak_uv": peak_to_peak,
        "maximum_step_uv": maximum_step,
        "failed_ptp": ~ptp_clean_mask,
        "failed_step": ~step_clean_mask,
        "failed_flat": ~non_flat_mask,
        "clean": clean_mask,
    })

    return {
        "quality_table": quality_table,
        "clean_mask": clean_mask,
        "ptp_threshold_uv": ptp_threshold,
        "step_threshold_uv": step_threshold,
    }


def classify_epoch_status(
    quality_table,
    participant,
    scene_code,
    channel,
    decision_table,
):
    table = quality_table.copy()

    decision_subset = decision_table.loc[
        (decision_table["participant"] == participant)
        & (decision_table["scene_code"] == scene_code)
        & (decision_table["channel"] == channel),
        ["epoch", "manual_decision"],
    ].copy()

    decision_lookup = dict(
        zip(decision_subset["epoch"], decision_subset["manual_decision"])
    )

    auto_reason = []
    status = []
    final_keep = []
    manual_decision_values = []

    for _, row in table.iterrows():
        epoch = int(row["epoch"])
        failed_ptp = bool(row["failed_ptp"])
        failed_step = bool(row["failed_step"])
        failed_flat = bool(row["failed_flat"])

        if failed_flat:
            automatic_reason = "flat"
            automatic_status = "auto_reject_flat"
            automatic_keep = False
        elif failed_ptp and failed_step:
            automatic_reason = "both"
            automatic_status = "auto_reject_both"
            automatic_keep = False
        elif failed_ptp:
            automatic_reason = "ptp_only"
            automatic_status = "auto_reject_ptp"
            automatic_keep = False
        elif failed_step:
            automatic_reason = "step_only"
            automatic_status = "auto_reject_step"
            automatic_keep = False
        else:
            automatic_reason = "kept"
            automatic_status = "clean"
            automatic_keep = True

        decision = decision_lookup.get(epoch, "")

        if decision == "keep":
            final_status = "manual_restored"
            keep_value = True
        elif decision == "reject":
            final_status = "manual_bad"
            keep_value = False
        else:
            final_status = automatic_status
            keep_value = automatic_keep

        auto_reason.append(automatic_reason)
        status.append(final_status)
        final_keep.append(keep_value)
        manual_decision_values.append(decision)

    table["auto_reason"] = auto_reason
    table["manual_decision"] = manual_decision_values
    table["status"] = status
    table["final_keep"] = final_keep
    return table


def frequency_mask(frequencies, low, high, include_high):
    if include_high:
        return (frequencies >= low) & (frequencies <= high)
    return (frequencies >= low) & (frequencies < high)


def integrate_band_power(psd_epochs, frequencies, low, high, include_high):
    mask = frequency_mask(frequencies, low, high, include_high)
    if mask.sum() < 2:
        raise ValueError(
            f"Insufficient Welch bins for band {low}-{high} Hz."
        )
    return trapezoid(
        psd_epochs[:, mask],
        frequencies[mask],
        axis=1,
    )


def sort_output_table(df, extra_columns=None):
    extra_columns = extra_columns or []
    sort_columns = ["participant", "scene_code"] + extra_columns

    return df.sort_values(
        sort_columns,
        key=lambda s: (
            s.map(SCENE_SORT_ORDER)
            if s.name == "scene_code"
            else s.map({p: i for i, p in enumerate(EXPECTED_PARTICIPANTS)})
            if s.name == "participant"
            else s
        ),
    ).reset_index(drop=True)



## 4. QC overview figure for each recording

In [5]:

# =========================================================
# 4. PER-RECORDING QC OVERVIEW
# =========================================================

def save_qc_overview(
    file_path,
    metadata,
    right_status_table,
    left_status_table,
    right_quality,
    left_quality,
):
    status_styles = {
        "clean": {
            "color": "#4c78a8",
            "marker": "o",
            "label": "Kept",
            "size": 18,
        },
        "auto_reject_ptp": {
            "color": "#e15759",
            "marker": "x",
            "label": "Auto reject: peak-to-peak only",
            "size": 55,
        },
        "auto_reject_step": {
            "color": "#f28e2b",
            "marker": "s",
            "label": "Auto reject: maximum-step only",
            "size": 38,
        },
        "auto_reject_both": {
            "color": "#b07aa1",
            "marker": "D",
            "label": "Auto reject: both rules",
            "size": 42,
        },
        "auto_reject_flat": {
            "color": "#8c564b",
            "marker": "v",
            "label": "Auto reject: flat",
            "size": 42,
        },
        "manual_bad": {
            "color": "#000000",
            "marker": "P",
            "label": "Manual reject",
            "size": 55,
        },
        "manual_restored": {
            "color": "#59a14f",
            "marker": "*",
            "label": "Manually restored",
            "size": 95,
        },
    }

    fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True)

    panel_items = [
        (
            axes[0, 0],
            right_status_table,
            "peak_to_peak_uv",
            right_quality["ptp_threshold_uv"],
            "Right-ear peak-to-peak",
            "Right PtP threshold",
        ),
        (
            axes[0, 1],
            right_status_table,
            "maximum_step_uv",
            right_quality["step_threshold_uv"],
            "Right-ear maximum-step",
            "Right step threshold",
        ),
        (
            axes[1, 0],
            left_status_table,
            "peak_to_peak_uv",
            left_quality["ptp_threshold_uv"],
            "Left-ear peak-to-peak",
            "Left PtP threshold",
        ),
        (
            axes[1, 1],
            left_status_table,
            "maximum_step_uv",
            left_quality["step_threshold_uv"],
            "Left-ear maximum-step",
            "Left step threshold",
        ),
    ]

    for ax, table, metric_col, threshold, title, threshold_label in panel_items:
        ax.plot(
            table["start_time_s"],
            table[metric_col],
            color="#c7c7c7",
            linewidth=0.8,
            zorder=1,
        )
        ax.axhline(
            threshold,
            color="#1f77b4",
            linestyle="--",
            linewidth=1.2,
            label=f"{threshold_label} ({threshold:.1f} µV)",
            zorder=2,
        )

        for status_name, style in status_styles.items():
            subset = table.loc[table["status"] == status_name]
            if subset.empty:
                continue
            ax.scatter(
                subset["start_time_s"],
                subset[metric_col],
                c=style["color"],
                marker=style["marker"],
                s=style["size"],
                label=style["label"],
                zorder=3,
            )

        ax.set_title(title)
        ax.set_ylabel("Amplitude (µV)")
        ax.grid(alpha=0.3)

    axes[1, 0].set_xlabel("Epoch start time after 15-s trim (s)")
    axes[1, 1].set_xlabel("Epoch start time after 15-s trim (s)")

    handles, labels = [], []
    for ax in axes.flat:
        h, l = ax.get_legend_handles_labels()
        for hh, ll in zip(h, l):
            if ll not in labels:
                handles.append(hh)
                labels.append(ll)

    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.93),
        ncol=3,
        frameon=True,
        fontsize=9,
    )

    summary_text = (
        f"Right kept: {int(right_status_table['final_keep'].sum())}/{len(right_status_table)} | "
        f"Left kept: {int(left_status_table['final_keep'].sum())}/{len(left_status_table)} | "
        f"Joint kept: {int((right_status_table['final_keep'].to_numpy() & left_status_table['final_keep'].to_numpy()).sum())}"
    )
    fig.text(0.01, 0.01, summary_text, ha="left", va="bottom", fontsize=10)

    fig.suptitle(
        f"{metadata['participant']} - {metadata['scene_name']} - EEG QC",
        fontsize=17,
        y=0.98,
    )

    plt.tight_layout(rect=[0, 0.05, 1, 0.84])

    output_path = QC_OVERVIEW_DIR / f"{file_path.stem}_QC_reason_coloured.png"
    plt.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(fig)



## 5. Process one recording

In [6]:

# =========================================================
# 5. ONE-RECORDING PROCESSOR
# =========================================================

def process_one_file(file_path: Path):
    metadata = parse_filename(file_path)
    df = load_recording(file_path)

    right_raw_v = df[RIGHT_EEG_COL].to_numpy(dtype=float)
    left_raw_v = df[LEFT_EEG_COL].to_numpy(dtype=float)

    original_duration_s = len(df) / FS

    start_sample = int(round(START_TRIM_SECONDS * FS))
    if END_TRIM_SECONDS > 0:
        end_sample = len(df) - int(round(END_TRIM_SECONDS * FS))
    else:
        end_sample = len(df)

    if start_sample >= end_sample:
        raise ValueError(f"{file_path.name}: trim removes the entire recording.")

    right_analysis_raw_v = right_raw_v[start_sample:end_sample]
    left_analysis_raw_v = left_raw_v[start_sample:end_sample]
    trimmed_duration_s = len(right_analysis_raw_v) / FS

    right_filtered_uv = (
        bandpass_filter(
            right_analysis_raw_v,
            FS,
            EEG_BAND[0],
            EEG_BAND[1],
            FILTER_ORDER,
        )
        * V_TO_UV
    )

    left_filtered_uv = (
        bandpass_filter(
            left_analysis_raw_v,
            FS,
            EEG_BAND[0],
            EEG_BAND[1],
            FILTER_ORDER,
        )
        * V_TO_UV
    )

    right_epochs = create_epochs(right_filtered_uv, FS, EPOCH_SECONDS)
    left_epochs = create_epochs(left_filtered_uv, FS, EPOCH_SECONDS)

    if right_epochs.shape != left_epochs.shape:
        raise ValueError(f"{file_path.name}: right/left epoch shapes differ.")

    number_of_epochs = right_epochs.shape[0]

    right_quality = assess_epoch_quality(
        right_epochs, EPOCH_SECONDS, NUMBER_OF_MADS
    )
    left_quality = assess_epoch_quality(
        left_epochs, EPOCH_SECONDS, NUMBER_OF_MADS
    )

    right_status = classify_epoch_status(
        right_quality["quality_table"],
        metadata["participant"],
        metadata["scene_code"],
        "right",
        MANUAL_DECISIONS,
    )
    left_status = classify_epoch_status(
        left_quality["quality_table"],
        metadata["participant"],
        metadata["scene_code"],
        "left",
        MANUAL_DECISIONS,
    )

    right_raw_epochs_uv = create_epochs(
        right_analysis_raw_v * V_TO_UV, FS, EPOCH_SECONDS
    )
    left_raw_epochs_uv = create_epochs(
        left_analysis_raw_v * V_TO_UV, FS, EPOCH_SECONDS
    )

    for status_table, raw_epochs_uv in [
        (right_status, right_raw_epochs_uv),
        (left_status, left_raw_epochs_uv),
    ]:
        raw_ptp = np.ptp(raw_epochs_uv, axis=1)
        raw_step = np.max(np.abs(np.diff(raw_epochs_uv, axis=1)), axis=1)

        raw_ptp_neighbour = neighbour_median(raw_ptp)
        raw_step_neighbour = neighbour_median(raw_step)

        status_table["raw_peak_to_peak_uv"] = raw_ptp
        status_table["raw_maximum_step_uv"] = raw_step
        status_table["raw_ptp_neighbour_median_uv"] = raw_ptp_neighbour
        status_table["raw_step_neighbour_median_uv"] = raw_step_neighbour
        status_table["raw_ptp_ratio_to_neighbours"] = np.divide(
            raw_ptp,
            raw_ptp_neighbour,
            out=np.full(raw_ptp.shape, np.nan, dtype=float),
            where=np.isfinite(raw_ptp_neighbour) & (raw_ptp_neighbour != 0),
        )

        status_table["raw_step_ratio_to_neighbours"] = np.divide(
            raw_step,
            raw_step_neighbour,
            out=np.full(raw_step.shape, np.nan, dtype=float),
            where=np.isfinite(raw_step_neighbour) & (raw_step_neighbour != 0),
        )

    # Add identifiers and fixed thresholds to the channel-level status tables.
    for channel_name, status_table, quality in [
        ("right", right_status, right_quality),
        ("left", left_status, left_quality),
    ]:
        status_table["participant"] = metadata["participant"]
        status_table["scene_code"] = metadata["scene_code"]
        status_table["scene_name"] = metadata["scene_name"]
        status_table["file_name"] = file_path.name
        status_table["channel"] = channel_name
        status_table["ptp_threshold_uv"] = quality["ptp_threshold_uv"]
        status_table["step_threshold_uv"] = quality["step_threshold_uv"]
        status_table["filtered_ptp_ratio"] = (
            status_table["peak_to_peak_uv"] / status_table["ptp_threshold_uv"]
        )
        status_table["filtered_step_ratio"] = (
            status_table["maximum_step_uv"] / status_table["step_threshold_uv"]
        )
        status_table["review_id"] = (
            status_table["participant"]
            + "_"
            + status_table["scene_code"]
            + "_"
            + channel_name
            + "_epoch"
            + status_table["epoch"].astype(str)
        )

    final_right_clean_mask = right_status["final_keep"].to_numpy(dtype=bool)
    final_left_clean_mask = left_status["final_keep"].to_numpy(dtype=bool)
    final_joint_clean_mask = final_right_clean_mask & final_left_clean_mask

    clean_epoch_indices = np.flatnonzero(final_joint_clean_mask)

    if len(clean_epoch_indices) == 0:
        raise ValueError(f"{file_path.name}: no jointly clean epochs remain.")

    right_clean_epochs = right_epochs[final_joint_clean_mask]
    left_clean_epochs = left_epochs[final_joint_clean_mask]

    welch_nperseg = int(WELCH_WINDOW_SECONDS * FS)
    welch_noverlap = welch_nperseg // 2

    frequencies, right_psd_epochs = signal.welch(
        right_clean_epochs,
        fs=FS,
        window="hann",
        nperseg=welch_nperseg,
        noverlap=welch_noverlap,
        detrend="constant",
        scaling="density",
        axis=1,
    )

    _, left_psd_epochs = signal.welch(
        left_clean_epochs,
        fs=FS,
        window="hann",
        nperseg=welch_nperseg,
        noverlap=welch_noverlap,
        detrend="constant",
        scaling="density",
        axis=1,
    )

    total_mask = (
        (frequencies >= TOTAL_POWER_BAND[0])
        & (frequencies <= TOTAL_POWER_BAND[1])
    )

    right_total_power = trapezoid(
        right_psd_epochs[:, total_mask],
        frequencies[total_mask],
        axis=1,
    )
    left_total_power = trapezoid(
        left_psd_epochs[:, total_mask],
        frequencies[total_mask],
        axis=1,
    )

    if np.any(right_total_power <= 0) or np.any(left_total_power <= 0):
        raise ValueError(f"{file_path.name}: non-positive 4–30 Hz total power.")

    epoch_result = pd.DataFrame({
        "participant": metadata["participant"],
        "scene_code": metadata["scene_code"],
        "scene_name": metadata["scene_name"],
        "file_name": file_path.name,
        "epoch": clean_epoch_indices,
        "start_time_after_trim_s": clean_epoch_indices * EPOCH_SECONDS,
        "end_time_after_trim_s": (clean_epoch_indices + 1) * EPOCH_SECONDS,
        "right_total_power_4_30_uv2": right_total_power,
        "left_total_power_4_30_uv2": left_total_power,
    })

    for band_name, definition in BAND_DEFINITIONS.items():
        right_power = integrate_band_power(
            right_psd_epochs,
            frequencies,
            definition["low"],
            definition["high"],
            definition["include_high"],
        )
        left_power = integrate_band_power(
            left_psd_epochs,
            frequencies,
            definition["low"],
            definition["high"],
            definition["include_high"],
        )

        epoch_result[f"right_absolute_{band_name}_uv2"] = right_power
        epoch_result[f"left_absolute_{band_name}_uv2"] = left_power

        epoch_result[f"right_relative_{band_name}"] = right_power / right_total_power
        epoch_result[f"left_relative_{band_name}"] = left_power / left_total_power

        epoch_result[f"bilateral_absolute_{band_name}_uv2"] = (
            right_power + left_power
        ) / 2

        epoch_result[f"bilateral_relative_{band_name}"] = (
            epoch_result[f"right_relative_{band_name}"]
            + epoch_result[f"left_relative_{band_name}"]
        ) / 2

    epoch_result["bilateral_total_power_4_30_uv2"] = (
        epoch_result["right_total_power_4_30_uv2"]
        + epoch_result["left_total_power_4_30_uv2"]
    ) / 2

    # Recording-level spectral-shape table.
    # Normalize each epoch PSD by its own 4–30 Hz power before taking the median.
    right_relative_psd_epochs = right_psd_epochs / right_total_power[:, None]
    left_relative_psd_epochs = left_psd_epochs / left_total_power[:, None]

    psd_keep_mask = (
        (frequencies >= EEG_BAND[0])
        & (frequencies <= EEG_BAND[1])
    )

    psd_table = pd.DataFrame({
        "participant": metadata["participant"],
        "scene_code": metadata["scene_code"],
        "scene_name": metadata["scene_name"],
        "file_name": file_path.name,
        "frequency_hz": frequencies[psd_keep_mask],
        "right_absolute_psd_median_uv2_per_hz": np.median(
            right_psd_epochs[:, psd_keep_mask], axis=0
        ),
        "left_absolute_psd_median_uv2_per_hz": np.median(
            left_psd_epochs[:, psd_keep_mask], axis=0
        ),
        "right_relative_psd_median_per_hz": np.median(
            right_relative_psd_epochs[:, psd_keep_mask], axis=0
        ),
        "left_relative_psd_median_per_hz": np.median(
            left_relative_psd_epochs[:, psd_keep_mask], axis=0
        ),
    })

    psd_table["bilateral_absolute_psd_median_uv2_per_hz"] = (
        psd_table["right_absolute_psd_median_uv2_per_hz"]
        + psd_table["left_absolute_psd_median_uv2_per_hz"]
    ) / 2

    psd_table["bilateral_relative_psd_median_per_hz"] = (
        psd_table["right_relative_psd_median_per_hz"]
        + psd_table["left_relative_psd_median_per_hz"]
    ) / 2

    summary = {
        "participant": metadata["participant"],
        "scene_code": metadata["scene_code"],
        "scene_name": metadata["scene_name"],
        "file_name": file_path.name,
        "clean_epochs": len(epoch_result),
        "usable_duration_s": len(epoch_result) * EPOCH_SECONDS,
        "joint_retention_percent": final_joint_clean_mask.mean() * 100,
    }

    for channel in ["right", "left", "bilateral"]:
        for band in BAND_DEFINITIONS:
            rel_col = f"{channel}_relative_{band}"
            abs_col = f"{channel}_absolute_{band}_uv2"

            summary[f"{channel}_relative_{band}_median"] = (
                epoch_result[rel_col].median()
            )
            summary[f"{channel}_relative_{band}_median_percent"] = (
                100 * summary[f"{channel}_relative_{band}_median"]
            )
            summary[f"{channel}_absolute_{band}_median_uv2"] = (
                epoch_result[abs_col].median()
            )

    for channel in ["right", "left", "bilateral"]:
        summary[f"{channel}_total_power_4_30_median_uv2"] = (
            epoch_result[f"{channel}_total_power_4_30_uv2"].median()
        )

    def status_count(table, status_name):
        return int((table["status"] == status_name).sum())

    qc = {
        "participant": metadata["participant"],
        "scene_code": metadata["scene_code"],
        "scene_name": metadata["scene_name"],
        "file_name": file_path.name,
        "original_duration_s": original_duration_s,
        "trimmed_duration_s": trimmed_duration_s,
        "total_epochs": number_of_epochs,
        "right_clean_epochs": int(final_right_clean_mask.sum()),
        "left_clean_epochs": int(final_left_clean_mask.sum()),
        "joint_clean_epochs": int(final_joint_clean_mask.sum()),
        "joint_retention_percent": final_joint_clean_mask.mean() * 100,
        "right_ptp_threshold_uv": right_quality["ptp_threshold_uv"],
        "right_step_threshold_uv": right_quality["step_threshold_uv"],
        "left_ptp_threshold_uv": left_quality["ptp_threshold_uv"],
        "left_step_threshold_uv": left_quality["step_threshold_uv"],
        "right_auto_ptp_only_count": int((right_status["auto_reason"] == "ptp_only").sum()),
        "right_auto_step_only_count": int((right_status["auto_reason"] == "step_only").sum()),
        "right_auto_both_count": int((right_status["auto_reason"] == "both").sum()),
        "right_auto_flat_count": int((right_status["auto_reason"] == "flat").sum()),
        "left_auto_ptp_only_count": int((left_status["auto_reason"] == "ptp_only").sum()),
        "left_auto_step_only_count": int((left_status["auto_reason"] == "step_only").sum()),
        "left_auto_both_count": int((left_status["auto_reason"] == "both").sum()),
        "left_auto_flat_count": int((left_status["auto_reason"] == "flat").sum()),
        "right_manual_bad_count": status_count(right_status, "manual_bad"),
        "right_manual_restored_count": status_count(right_status, "manual_restored"),
        "left_manual_bad_count": status_count(left_status, "manual_bad"),
        "left_manual_restored_count": status_count(left_status, "manual_restored"),
        "final_rejected_right_epochs": ", ".join(
            map(str, right_status.loc[~right_status["final_keep"], "epoch"].tolist())
        ),
        "final_rejected_left_epochs": ", ".join(
            map(str, left_status.loc[~left_status["final_keep"], "epoch"].tolist())
        ),
    }

    qc["qc_review_needed"] = (
        qc["joint_retention_percent"] < 80.0
        or qc["right_auto_step_only_count"] > 0
        or qc["left_auto_step_only_count"] > 0
        or qc["right_auto_ptp_only_count"] > 0
        or qc["left_auto_ptp_only_count"] > 0
        or qc["right_manual_bad_count"] > 0
        or qc["left_manual_bad_count"] > 0
        or qc["right_manual_restored_count"] > 0
        or qc["left_manual_restored_count"] > 0
    )

    if SAVE_QC_OVERVIEWS:
        save_qc_overview(
            file_path,
            metadata,
            right_status,
            left_status,
            right_quality,
            left_quality,
        )

    return summary, qc, epoch_result, right_status, left_status, psd_table


## 6. Process all 66 recordings

In [7]:

# =========================================================
# 6. PROCESS ALL RECORDINGS
# =========================================================

scene_rows = []
qc_rows = []
epoch_tables = []
status_tables = []
psd_tables = []
errors = []

print("Processing the complete 66-recording dataset...")

for number, file_path in enumerate(INPUT_FILES, start=1):
    print(f"[{number:02d}/{len(INPUT_FILES)}] {file_path.name}")

    try:
        (
            scene_row,
            qc_row,
            epoch_table,
            right_status,
            left_status,
            psd_table,
        ) = process_one_file(file_path)

        scene_rows.append(scene_row)
        qc_rows.append(qc_row)
        epoch_tables.append(epoch_table)
        status_tables.extend([right_status, left_status])
        psd_tables.append(psd_table)

        print(
            f"    joint clean epochs = {scene_row['clean_epochs']}; "
            f"retention = {scene_row['joint_retention_percent']:.1f}%"
        )

    except Exception as exc:
        errors.append({
            "file_name": file_path.name,
            "error": str(exc),
        })
        print("    ERROR:", exc)

if errors:
    error_table = pd.DataFrame(errors)
    display(error_table)
    error_path = OUTPUT_DIR / "script3_processing_errors.csv"
    error_table.to_csv(error_path, index=False)
    raise RuntimeError(
        "At least one recording failed. Fix the errors before using combined results."
    )

scene_eeg_summary_all = sort_output_table(pd.DataFrame(scene_rows))
qc_summary_all = sort_output_table(pd.DataFrame(qc_rows))

epoch_band_results_all = pd.concat(epoch_tables, ignore_index=True)
epoch_band_results_all = sort_output_table(
    epoch_band_results_all,
    extra_columns=["epoch"],
)

status_table_all = pd.concat(status_tables, ignore_index=True)
status_table_all = sort_output_table(
    status_table_all,
    extra_columns=["channel", "epoch"],
)

psd_by_recording_all = pd.concat(psd_tables, ignore_index=True)
psd_by_recording_all = sort_output_table(
    psd_by_recording_all,
    extra_columns=["frequency_hz"],
)

print("\nCompleted all recordings successfully.")
print("Scene-summary rows:", len(scene_eeg_summary_all))
print("Expected scene-summary rows:", EXPECTED_NUMBER_OF_FILES)


Processing the complete 66-recording dataset...
[01/66] IC01_BG_EEG.csv
    joint clean epochs = 51; retention = 73.9%
[02/66] IC01_G_EEG.csv
    joint clean epochs = 69; retention = 97.2%
[03/66] IC01_GR_EEG.csv
    joint clean epochs = 71; retention = 100.0%
[04/66] IC02_BG_EEG.csv
    joint clean epochs = 66; retention = 95.7%
[05/66] IC02_G_EEG.csv
    joint clean epochs = 70; retention = 100.0%
[06/66] IC02_GR_EEG.csv
    joint clean epochs = 67; retention = 94.4%
[07/66] IC03_BG_EEG.csv
    joint clean epochs = 69; retention = 97.2%
[08/66] IC03_G_EEG.csv
    joint clean epochs = 70; retention = 98.6%
[09/66] IC03_GR_EEG.csv
    joint clean epochs = 71; retention = 100.0%
[10/66] IC04_BG_EEG.csv
    joint clean epochs = 71; retention = 100.0%
[11/66] IC04_G_EEG.csv
    joint clean epochs = 71; retention = 100.0%
[12/66] IC04_GR_EEG.csv
    joint clean epochs = 67; retention = 94.4%
[13/66] IC05_BG_EEG.csv
    joint clean epochs = 68; retention = 95.8%
[14/66] IC05_G_EEG.csv
    j

## 7. Build/preserve the independent manual-review decision table

In [8]:

# =========================================================
# 7. REVIEW QUEUE + DECISION TABLE
# =========================================================

review_candidates = status_table_all.loc[
    status_table_all["auto_reason"] != "kept"
].copy()

review_candidates["review_priority"] = (
    review_candidates["auto_reason"].map(REVIEW_PRIORITY_MAP)
)

decision_columns = [
    "participant",
    "scene_code",
    "scene_name",
    "file_name",
    "channel",
    "epoch",
    "review_id",
    "review_priority",
    "auto_reason",
    "peak_to_peak_uv",
    "maximum_step_uv",
    "filtered_ptp_ratio",
    "filtered_step_ratio",
    "raw_peak_to_peak_uv",
    "raw_maximum_step_uv",
    "raw_ptp_ratio_to_neighbours",
    "raw_step_ratio_to_neighbours",
]

base_decision_table = review_candidates[decision_columns].copy()

# Preserve previous decisions and allow manually added rejections of auto-kept epochs.
previous = MANUAL_DECISIONS.copy()

previous_small_columns = [
    "participant",
    "scene_code",
    "channel",
    "epoch",
    "manual_decision",
    "review_note",
    "reviewer",
]
previous_small = previous[previous_small_columns].copy()

decision_table = base_decision_table.merge(
    previous_small,
    on=["participant", "scene_code", "channel", "epoch"],
    how="left",
    validate="one_to_one",
)

for column in ["manual_decision", "review_note", "reviewer"]:
    decision_table[column] = decision_table[column].fillna("").astype(str)

base_keys = set(
    map(
        tuple,
        base_decision_table[
            ["participant", "scene_code", "channel", "epoch"]
        ].to_numpy(),
    )
)

if not previous_small.empty:
    previous_keys = list(
        map(
            tuple,
            previous_small[
                ["participant", "scene_code", "channel", "epoch"]
            ].to_numpy(),
        )
    )
    extra_previous = previous_small.loc[
        [key not in base_keys for key in previous_keys]
    ].copy()
else:
    extra_previous = pd.DataFrame(columns=previous_small_columns)

if not extra_previous.empty:
    extra_rows = status_table_all.merge(
        extra_previous,
        on=["participant", "scene_code", "channel", "epoch"],
        how="inner",
        validate="one_to_one",
        suffixes=("_status", ""),
    )

    if len(extra_rows) != len(extra_previous):
        raise ValueError(
            "One or more manually added decision rows do not match the current EEG data."
        )

    extra_rows["review_priority"] = "manual"
    extra_rows["review_id"] = (
        extra_rows["participant"]
        + "_"
        + extra_rows["scene_code"]
        + "_"
        + extra_rows["channel"]
        + "_epoch"
        + extra_rows["epoch"].astype(str)
    )

    # Normalize names created by merge.
    if "scene_name_status" in extra_rows.columns:
        extra_rows["scene_name"] = extra_rows["scene_name_status"]
    if "file_name_status" in extra_rows.columns:
        extra_rows["file_name"] = extra_rows["file_name_status"]

    for column in decision_table.columns:
        if column not in extra_rows.columns:
            extra_rows[column] = ""

    decision_table = pd.concat(
        [decision_table, extra_rows[decision_table.columns]],
        ignore_index=True,
    )

decision_table = sort_output_table(
    decision_table,
    extra_columns=["channel", "epoch"],
)

# A mandatory candidate is considered reviewed when either:
# - reviewer is filled in (meaning the automatic rejection was inspected and accepted), or
# - manual_decision explicitly contains keep/reject.
decision_table["review_completed"] = (
    decision_table["reviewer"].astype(str).str.strip().ne("")
    | decision_table["manual_decision"].astype(str).str.strip().ne("")
)

mandatory_mask = decision_table["review_priority"].isin(
    MANDATORY_REVIEW_PRIORITIES
)

pending_review = decision_table.loc[
    mandatory_mask & ~decision_table["review_completed"]
].copy()

decision_table.to_csv(DECISION_FILE, index=False)

review_queue_csv = OUTPUT_DIR / "script3_qc_review_candidates.csv"
review_candidates.to_csv(review_queue_csv, index=False)

print("Automatic review candidates:", len(review_candidates))
print(
    "High priority:",
    int((decision_table["review_priority"] == "high").sum()),
)
print(
    "Medium priority:",
    int((decision_table["review_priority"] == "medium").sum()),
)
print(
    "Low priority:",
    int((decision_table["review_priority"] == "low").sum()),
)
print("Pending mandatory high/medium review:", len(pending_review))

if len(pending_review) > 0:
    display(
        pending_review[
            [
                "participant",
                "scene_code",
                "channel",
                "epoch",
                "review_priority",
                "auto_reason",
                "filtered_ptp_ratio",
                "filtered_step_ratio",
            ]
        ].head(30)
    )


Automatic review candidates: 498
High priority: 159
Medium priority: 12
Low priority: 327
Pending mandatory high/medium review: 0



## 8. Detailed review plots for pending high/medium candidates

The layout reproduces the independent QC-review logic:

- filtered previous epoch;
- filtered selected epoch;
- filtered next epoch;
- centred raw signal with ±1 s context.

**How to record review:**

- if automatic rejection is correct: leave `manual_decision` blank and fill `reviewer`;
- if rejected epoch should be restored: `manual_decision = keep`;
- if an automatically kept neighbouring epoch should be manually excluded, add a row to the decision CSV with `manual_decision = reject`.

After editing the CSV, restart the kernel and **Run All** again.


In [9]:

# =========================================================
# 8. DETAILED REVIEW-PLOT EXPORT
# =========================================================

def prepare_review_recording(file_path: Path):
    df = load_recording(file_path)

    start_sample = int(round(START_TRIM_SECONDS * FS))
    if END_TRIM_SECONDS > 0:
        end_sample = len(df) - int(round(END_TRIM_SECONDS * FS))
    else:
        end_sample = len(df)

    right_raw_v = df[RIGHT_EEG_COL].to_numpy(dtype=float)[start_sample:end_sample]
    left_raw_v = df[LEFT_EEG_COL].to_numpy(dtype=float)[start_sample:end_sample]

    right_filtered_uv = (
        bandpass_filter(
            right_raw_v,
            FS,
            EEG_BAND[0],
            EEG_BAND[1],
            FILTER_ORDER,
        )
        * V_TO_UV
    )
    left_filtered_uv = (
        bandpass_filter(
            left_raw_v,
            FS,
            EEG_BAND[0],
            EEG_BAND[1],
            FILTER_ORDER,
        )
        * V_TO_UV
    )

    return {
        "right_raw_v": right_raw_v,
        "left_raw_v": left_raw_v,
        "right_epochs": create_epochs(right_filtered_uv, FS, EPOCH_SECONDS),
        "left_epochs": create_epochs(left_filtered_uv, FS, EPOCH_SECONDS),
    }


def save_detailed_review_plot(candidate_row, review_data):
    participant = candidate_row["participant"]
    scene_code = candidate_row["scene_code"]
    channel = candidate_row["channel"]
    epoch_index = int(candidate_row["epoch"])
    review_id = candidate_row["review_id"]

    if channel == "right":
        epochs = review_data["right_epochs"]
        raw_signal_v = review_data["right_raw_v"]
        channel_name = "Right-ear EEG"
    elif channel == "left":
        epochs = review_data["left_epochs"]
        raw_signal_v = review_data["left_raw_v"]
        channel_name = "Left-ear EEG"
    else:
        raise ValueError("channel must be 'right' or 'left'.")

    qc_table = status_table_all.loc[
        (status_table_all["participant"] == participant)
        & (status_table_all["scene_code"] == scene_code)
        & (status_table_all["channel"] == channel)
    ].copy()

    selected_indices = [
        index
        for index in [epoch_index - 1, epoch_index, epoch_index + 1]
        if 0 <= index < len(epochs)
    ]

    fig, axes = plt.subplots(
        len(selected_indices) + 1,
        1,
        figsize=(13, 2.65 * (len(selected_indices) + 1)),
    )

    axes = np.atleast_1d(axes)

    for ax, current_epoch in zip(axes[:-1], selected_indices):
        current_signal = epochs[current_epoch]
        start_time = current_epoch * EPOCH_SECONDS
        time_axis = start_time + np.arange(current_signal.size) / FS

        current_row = qc_table.loc[
            qc_table["epoch"] == current_epoch
        ].iloc[0]

        position_text = "SELECTED" if current_epoch == epoch_index else "Neighbour"
        linewidth = 1.3 if current_epoch == epoch_index else 0.9

        ax.plot(time_axis, current_signal, linewidth=linewidth)
        ax.axhline(0, linestyle="--", linewidth=0.7, alpha=0.7)

        ax.set_title(
            f"{position_text} — epoch {current_epoch}: "
            f"{start_time:.0f}–{start_time + EPOCH_SECONDS:.0f} s"
            f" | PtP={current_row['peak_to_peak_uv']:.1f} µV"
            f" | Step={current_row['maximum_step_uv']:.1f} µV"
            f" | Reason={current_row['auto_reason']}"
        )
        ax.set_ylabel("Filtered amplitude (µV)")
        ax.grid(alpha=0.3)

    epoch_start_s = epoch_index * EPOCH_SECONDS
    epoch_end_s = epoch_start_s + EPOCH_SECONDS
    context_start_s = max(0, epoch_start_s - 1)
    context_end_s = min(len(raw_signal_v) / FS, epoch_end_s + 1)

    context_start_sample = int(round(context_start_s * FS))
    context_end_sample = int(round(context_end_s * FS))

    raw_time_axis = (
        np.arange(context_start_sample, context_end_sample) / FS
    )
    raw_segment_uv = (
        raw_signal_v[context_start_sample:context_end_sample] * V_TO_UV
    )
    raw_segment_uv = raw_segment_uv - np.median(raw_segment_uv)

    raw_ax = axes[-1]
    raw_ax.plot(raw_time_axis, raw_segment_uv, linewidth=0.8)
    raw_ax.axvspan(
        epoch_start_s,
        epoch_end_s,
        alpha=0.15,
        label=f"Epoch {epoch_index}",
    )
    raw_ax.set_title("Raw signal with one-second context")
    raw_ax.set_xlabel("Time after 15-s trim (s)")
    raw_ax.set_ylabel("Centred raw amplitude (µV)")
    raw_ax.grid(alpha=0.3)
    raw_ax.legend()

    fig.suptitle(
        f"{participant} - {SCENE_NAMES[scene_code]} - {channel_name}\n"
        f"{candidate_row['review_priority'].upper()} review priority | "
        f"{candidate_row['auto_reason']} | epoch {epoch_index}",
        fontsize=15,
        y=0.995,
    )

    plt.tight_layout()

    output_path = REVIEW_PLOT_DIR / f"{review_id}_review.png"
    plt.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(fig)

    return output_path


if GENERATE_REVIEW_PLOTS_FOR_PENDING and not pending_review.empty:
    review_cache = {}
    generated_paths = []

    print(f"Generating {len(pending_review)} pending review plots...")

    for _, candidate in pending_review.iterrows():
        file_name = candidate["file_name"]

        if file_name not in review_cache:
            file_path = DATA_DIR / file_name
            review_cache[file_name] = prepare_review_recording(file_path)

        generated_paths.append(
            save_detailed_review_plot(
                candidate,
                review_cache[file_name],
            )
        )

    print("Saved pending review plots to:", REVIEW_PLOT_DIR)
else:
    print("No pending review plots need to be generated.")


No pending review plots need to be generated.


## 9. Export complete EEG/QC tables and the Script 4 input table

In [10]:

# =========================================================
# 9. EXPORT TABLES
# =========================================================

scene_csv = OUTPUT_DIR / "script3_scene_eeg_summary.csv"
qc_csv = OUTPUT_DIR / "script3_qc_summary.csv"
epoch_csv = OUTPUT_DIR / "script3_epoch_band_results.csv"
status_csv = OUTPUT_DIR / "script3_epoch_qc_status.csv"
psd_csv = OUTPUT_DIR / "script3_psd_by_recording.csv"
script4_input_csv = OUTPUT_DIR / "script3_for_script4.csv"
excel_file = OUTPUT_DIR / "script3_EEG_results.xlsx"
manifest_csv = OUTPUT_DIR / "script3_run_manifest.csv"

scene_eeg_summary_all.to_csv(scene_csv, index=False)
qc_summary_all.to_csv(qc_csv, index=False)
epoch_band_results_all.to_csv(epoch_csv, index=False)
status_table_all.to_csv(status_csv, index=False)
psd_by_recording_all.to_csv(psd_csv, index=False)

# Compact scene-level table designed for Script 4 merge on participant + scene_code.
script4_input = scene_eeg_summary_all[
    [
        "participant",
        "scene_code",
        "scene_name",
        "file_name",
        "clean_epochs",
        "usable_duration_s",
        "joint_retention_percent",
        "bilateral_relative_theta_median_percent",
        "bilateral_relative_alpha_median_percent",
        "bilateral_relative_beta_median_percent",
        "bilateral_absolute_theta_median_uv2",
        "bilateral_absolute_alpha_median_uv2",
        "bilateral_absolute_beta_median_uv2",
        "bilateral_total_power_4_30_median_uv2",
        "right_relative_theta_median_percent",
        "right_relative_alpha_median_percent",
        "right_relative_beta_median_percent",
        "left_relative_theta_median_percent",
        "left_relative_alpha_median_percent",
        "left_relative_beta_median_percent",
    ]
].copy()

script4_input = script4_input.merge(
    qc_summary_all[
        [
            "participant",
            "scene_code",
            "qc_review_needed",
            "right_manual_bad_count",
            "right_manual_restored_count",
            "left_manual_bad_count",
            "left_manual_restored_count",
        ]
    ],
    on=["participant", "scene_code"],
    how="left",
    validate="one_to_one",
)

# Convenience names for the three primary bilateral relative-power features.
script4_input = script4_input.rename(columns={
    "bilateral_relative_theta_median_percent": "eeg_theta_relative_percent",
    "bilateral_relative_alpha_median_percent": "eeg_alpha_relative_percent",
    "bilateral_relative_beta_median_percent": "eeg_beta_relative_percent",
})

script4_input["source_script"] = SCRIPT_VERSION
script4_input.to_csv(script4_input_csv, index=False)

final_outputs_ready = (len(pending_review) == 0)

run_state = (
    "FINAL_REVIEW_COMPLETE"
    if final_outputs_ready
    else "PROVISIONAL_PENDING_QC_REVIEW"
)

manifest = pd.DataFrame([{
    "script_version": SCRIPT_VERSION,
    "run_timestamp": pd.Timestamp.now().isoformat(timespec="seconds"),
    "run_state": run_state,
    "expected_recordings": EXPECTED_NUMBER_OF_FILES,
    "processed_recordings": len(scene_eeg_summary_all),
    "participants": scene_eeg_summary_all["participant"].nunique(),
    "review_candidates_total": len(decision_table),
    "pending_mandatory_review": len(pending_review),
    "final_figures_ready": final_outputs_ready,
}])
manifest.to_csv(manifest_csv, index=False)

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    scene_eeg_summary_all.to_excel(writer, sheet_name="Scene summary", index=False)
    qc_summary_all.to_excel(writer, sheet_name="QC summary", index=False)
    epoch_band_results_all.to_excel(writer, sheet_name="Epoch results", index=False)
    status_table_all.to_excel(writer, sheet_name="Epoch QC status", index=False)
    psd_by_recording_all.to_excel(writer, sheet_name="PSD by recording", index=False)
    decision_table.to_excel(writer, sheet_name="Manual decisions", index=False)
    script4_input.to_excel(writer, sheet_name="Script4 input", index=False)
    manifest.to_excel(writer, sheet_name="Run manifest", index=False)

print("Run state:", run_state)
print("\nSaved:")
for path in [
    scene_csv,
    qc_csv,
    epoch_csv,
    status_csv,
    psd_csv,
    DECISION_FILE,
    script4_input_csv,
    excel_file,
    manifest_csv,
]:
    print(" -", path)



Run state: FINAL_REVIEW_COMPLETE

Saved:
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\script3_scene_eeg_summary.csv
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\script3_qc_summary.csv
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\script3_epoch_band_results.csv
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\script3_epoch_qc_status.csv
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\script3_psd_by_recording.csv
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\script3_manual_review_decisions.csv
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\script3_for_script4.csv
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\script3_EEG_results.xlsx
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\script3_run_manifest.csv



## 10. Final Script 3 figures

These figures are intentionally split into two purposes:

- **Reliability/QC:** retention and artifact-rejection composition.
- **EEG descriptive results:** within-participant relative band power and group relative PSD.

The environmental **inferential** analysis (including counterbalancing/order and multimodal triangulation) is reserved for Script 4.


In [11]:

# =========================================================
# 10. FINAL FIGURE FUNCTIONS
# =========================================================

def save_figure_1_qc_retention(scene_summary):
    pivot = scene_summary.pivot(
        index="participant",
        columns="scene_code",
        values="joint_retention_percent",
    ).reindex(columns=EXPECTED_SCENES)

    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    x = np.arange(len(EXPECTED_SCENES))

    for participant, row in pivot.iterrows():
        ax.plot(
            x,
            row.values.astype(float),
            color="#bdbdbd",
            linewidth=0.8,
            alpha=0.65,
            zorder=1,
        )
        for i, scene in enumerate(EXPECTED_SCENES):
            ax.scatter(
                i,
                row[scene],
                s=24,
                color=SCENE_COLORS[scene],
                alpha=0.8,
                zorder=2,
            )

    medians = pivot.median(axis=0)
    ax.scatter(
        x,
        medians.values,
        marker="D",
        s=85,
        color="black",
        label="Median",
        zorder=4,
    )

    ax.axhline(
        80,
        linestyle="--",
        linewidth=1.0,
        color="#555555",
        label="80% QC review threshold",
    )

    ax.set_xticks(x)
    ax.set_xticklabels([SCENE_NAMES[s] for s in EXPECTED_SCENES])
    ax.set_ylabel("Joint retained epochs (%)")
    ax.set_ylim(0, 103)
    ax.set_title("EEG data retention after final epoch QC")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)

    path = FIGURE_DIR / "Figure_S3_01_QC_retention.png"
    plt.tight_layout()
    plt.savefig(path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)
    return path


def save_figure_2_rejection_reasons(status_table):
    reason_order = ["ptp_only", "step_only", "both", "flat"]
    reason_labels = {
        "ptp_only": "Peak-to-peak only",
        "step_only": "Maximum-step only",
        "both": "Both rules",
        "flat": "Flat signal",
    }
    reason_colors = {
        "ptp_only": "#e15759",
        "step_only": "#f28e2b",
        "both": "#b07aa1",
        "flat": "#8c564b",
    }

    rows = []
    for scene in EXPECTED_SCENES:
        subset = status_table.loc[status_table["scene_code"] == scene]
        denominator = len(subset)

        row = {"scene_code": scene}
        for reason in reason_order:
            row[reason] = (
                100
                * (subset["auto_reason"] == reason).sum()
                / denominator
            )
        rows.append(row)

    plot_df = pd.DataFrame(rows).set_index("scene_code")
    plot_df = plot_df.reindex(EXPECTED_SCENES)

    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    x = np.arange(len(EXPECTED_SCENES))
    bottom = np.zeros(len(EXPECTED_SCENES))

    for reason in reason_order:
        values = plot_df[reason].to_numpy(dtype=float)
        ax.bar(
            x,
            values,
            bottom=bottom,
            label=reason_labels[reason],
            color=reason_colors[reason],
            width=0.62,
        )
        bottom = bottom + values

    ax.set_xticks(x)
    ax.set_xticklabels([SCENE_NAMES[s] for s in EXPECTED_SCENES])
    ax.set_ylabel("Automatically rejected channel-epochs (%)")
    ax.set_title("Composition of automatic EEG epoch rejections")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)

    path = FIGURE_DIR / "Figure_S3_02_QC_rejection_reasons.png"
    plt.tight_layout()
    plt.savefig(path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)
    return path


def save_figure_3_relative_band_power(scene_summary):
    band_titles = {
        "theta": "Theta (4–<8 Hz)",
        "alpha": "Alpha (8–<13 Hz)",
        "beta": "Beta (13–30 Hz)",
    }

    fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.6), sharex=True)
    x = np.arange(len(EXPECTED_SCENES))

    for ax, band in zip(axes, ["theta", "alpha", "beta"]):
        value_col = f"bilateral_relative_{band}_median_percent"

        pivot = scene_summary.pivot(
            index="participant",
            columns="scene_code",
            values=value_col,
        ).reindex(columns=EXPECTED_SCENES)

        for participant, row in pivot.iterrows():
            ax.plot(
                x,
                row.values.astype(float),
                color="#c7c7c7",
                linewidth=0.75,
                alpha=0.55,
                zorder=1,
            )

        data_for_boxes = [
            pivot[scene].dropna().to_numpy(dtype=float)
            for scene in EXPECTED_SCENES
        ]

        box = ax.boxplot(
            data_for_boxes,
            positions=x,
            widths=0.42,
            patch_artist=True,
            showfliers=False,
            medianprops={"color": "black", "linewidth": 1.3},
            whiskerprops={"linewidth": 1.0},
            capprops={"linewidth": 1.0},
        )

        for patch, scene in zip(box["boxes"], EXPECTED_SCENES):
            patch.set_facecolor(SCENE_COLORS[scene])
            patch.set_alpha(0.28)

        for i, scene in enumerate(EXPECTED_SCENES):
            values = pivot[scene].to_numpy(dtype=float)
            ax.scatter(
                np.full_like(values, i, dtype=float),
                values,
                s=24,
                color=SCENE_COLORS[scene],
                alpha=0.78,
                zorder=3,
            )

        ax.set_xticks(x)
        ax.set_xticklabels([SCENE_NAMES[s] for s in EXPECTED_SCENES], rotation=15)
        ax.set_title(band_titles[band])
        ax.grid(axis="y", alpha=0.22)

    axes[0].set_ylabel("Median relative band power (% of 4–30 Hz power)")
    fig.suptitle(
        "Within-participant EEG relative band power across environmental scenes",
        fontsize=16,
        y=1.02,
    )

    path = FIGURE_DIR / "Figure_S3_03_relative_band_power.png"
    plt.tight_layout()
    plt.savefig(path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)
    return path


def save_figure_4_group_relative_psd(psd_table):
    fig, ax = plt.subplots(figsize=(10.5, 6.4))

    for scene in EXPECTED_SCENES:
        subset = psd_table.loc[
            (psd_table["scene_code"] == scene)
            & (psd_table["frequency_hz"] >= 4.0)
            & (psd_table["frequency_hz"] <= 30.0)
        ].copy()

        grouped = subset.groupby("frequency_hz")[
            "bilateral_relative_psd_median_per_hz"
        ]

        mean_curve = grouped.mean()
        sem_curve = grouped.sem()

        frequencies = mean_curve.index.to_numpy(dtype=float)
        mean_values = 100 * mean_curve.to_numpy(dtype=float)
        sem_values = 100 * sem_curve.to_numpy(dtype=float)

        ax.plot(
            frequencies,
            mean_values,
            linewidth=2.0,
            color=SCENE_COLORS[scene],
            label=SCENE_NAMES[scene],
        )
        ax.fill_between(
            frequencies,
            mean_values - sem_values,
            mean_values + sem_values,
            color=SCENE_COLORS[scene],
            alpha=0.16,
        )

    ax.axvline(8, linestyle="--", linewidth=0.8, color="#777777")
    ax.axvline(13, linestyle="--", linewidth=0.8, color="#777777")

    ax.text(5.8, 0.98, "Theta", transform=ax.get_xaxis_transform(), ha="center", va="top")
    ax.text(10.5, 0.98, "Alpha", transform=ax.get_xaxis_transform(), ha="center", va="top")
    ax.text(21.5, 0.98, "Beta", transform=ax.get_xaxis_transform(), ha="center", va="top")

    ax.set_xlim(4, 30)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Relative PSD (% of 4–30 Hz power per Hz)")
    ax.set_title("Group bilateral relative power spectral density")
    ax.grid(alpha=0.22)
    ax.legend(frameon=False)

    path = FIGURE_DIR / "Figure_S3_04_group_relative_PSD.png"
    plt.tight_layout()
    plt.savefig(path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)
    return path


def save_supplementary_total_power(scene_summary):
    value_col = "bilateral_total_power_4_30_median_uv2"

    pivot = scene_summary.pivot(
        index="participant",
        columns="scene_code",
        values=value_col,
    ).reindex(columns=EXPECTED_SCENES)

    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    x = np.arange(len(EXPECTED_SCENES))

    for participant, row in pivot.iterrows():
        values = row.values.astype(float)

        ax.plot(
            x,
            values,
            color="#c7c7c7",
            linewidth=0.8,
            alpha=0.6,
        )

        for i, scene in enumerate(EXPECTED_SCENES):
            ax.scatter(
                i,
                row[scene],
                s=24,
                color=SCENE_COLORS[scene],
                alpha=0.78,
            )

    ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels([SCENE_NAMES[s] for s in EXPECTED_SCENES])
    ax.set_ylabel("Median bilateral total 4–30 Hz power (µV², log scale)")
    ax.set_title("Supplementary EEG absolute-power quality overview")
    ax.grid(axis="y", alpha=0.22)

    path = FIGURE_DIR / "Supplementary_S3_01_total_power_log.png"
    plt.tight_layout()
    plt.savefig(path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)
    return path


In [12]:

# =========================================================
# 11. CREATE FINAL FIGURES WHEN REVIEW IS COMPLETE
# =========================================================

if REQUIRE_REVIEW_BEFORE_FINAL_FIGURES and not final_outputs_ready:
    print(
        "FINAL FIGURES NOT GENERATED YET.\n"
        f"{len(pending_review)} high/medium QC candidates still need independent review.\n"
        f"Edit: {DECISION_FILE}\n"
        "Then restart the kernel and Run All."
    )

else:
    figure_paths = [
        save_figure_1_qc_retention(scene_eeg_summary_all),
        save_figure_2_rejection_reasons(status_table_all),
        save_figure_3_relative_band_power(scene_eeg_summary_all),
        save_figure_4_group_relative_psd(psd_by_recording_all),
        save_supplementary_total_power(scene_eeg_summary_all),
    ]

    print("Final Script 3 figures created:")
    for path in figure_paths:
        print(" -", path)


Final Script 3 figures created:
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\final_figures\Figure_S3_01_QC_retention.png
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\final_figures\Figure_S3_02_QC_rejection_reasons.png
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\final_figures\Figure_S3_03_relative_band_power.png
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\final_figures\Figure_S3_04_group_relative_PSD.png
 - C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\final_figures\Supplementary_S3_01_total_power_log.png



## 12. End-of-run summary

### Main outputs for Results / Discussion

- `script3_qc_summary.csv` → recording-level reliability and retention.
- `script3_scene_eeg_summary.csv` → 66 participant × scene EEG feature rows.
- `script3_epoch_band_results.csv` → retained-epoch feature table.
- `script3_psd_by_recording.csv` → spectral-shape table for group PSD plots.
- `script3_epoch_qc_status.csv` → complete audit trail of every channel-epoch QC decision.
- `script3_manual_review_decisions.csv` → external manual-review record.
- `script3_for_script4.csv` → compact 66-row scene-level EEG table for Script 4.
- `script3_EEG_results.xlsx` → combined workbook.
- `final_figures/` → final EEG/QC figures after mandatory review is complete.

### Interpretation boundary

The bilateral relative theta/alpha/beta summaries are the primary compact EEG features for later triangulation. Right/left values remain in the output for sensitivity checks, but the two-ear difference should **not** be interpreted as conventional frontal alpha asymmetry.

The Script 4 merge should preserve measurement level:

- **scene level (66 rows):** EEG, ECG, environment, trial order/position;
- **participant level (22 rows):** pre/post POMS change and participant metadata.

Do not count a participant-level POMS change three times as if it were three independent scene-specific POMS outcomes.
